<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_2_Advanced_NumPy/Day_11_NumPy_Applications.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 11: NumPy for Real Data Science Applications

## Phase 1 | Month 2 | Week 2 | Day 11

**Goal:** Apply NumPy to image processing, financial time series, and a complete ML preprocessing pipeline.

### What You Will Learn
1. Image processing: brightness, contrast, edge detection, pooling
2. Rolling windows for financial time series
3. Full preprocessing pipeline
4. Logistic regression from scratch with NumPy

### Resources
- 📖 [NumPy for Images — SciKit-Image](https://scikit-image.org/docs/stable/user_guide/numpy_images.html)
- 🎥 [Signal Processing with NumPy](https://www.youtube.com/watch?v=V0D2mhVt7NE)
- 📖 [Python Data Science Handbook — Ch 2](https://jakevdp.github.io/PythonDataScienceHandbook/)

In [ ]:
import numpy as np

# IMAGE PROCESSING WITH NUMPY
img = np.array([
    [ 20,  80, 120, 200, 150,  60],
    [ 30,  90, 130, 180, 140,  70],
    [ 25, 100, 200, 255, 160,  80],
    [ 15,  70,  90, 170, 130,  50],
    [ 10,  60,  80, 140, 100,  40],
    [  5,  50,  70, 100,  90,  30],
], dtype=np.uint8)

print('Original image:\n', img)

# Brightness adjustment
bright = np.clip(img.astype(np.int32) + 50, 0, 255).astype(np.uint8)
print('\nBrightened (first row):', bright[0])

# Contrast stretching
stretched = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
print('Contrast stretched (first row):', stretched[0])

# Horizontal edge detection (Sobel-like)
edges = np.abs(img[:, 1:].astype(np.int32) - img[:, :-1].astype(np.int32))
print('\nEdge magnitudes (first row):', edges[0])

# 2x2 Average Pooling (used in CNN architectures)
h, w = img.shape
pooled = img.reshape(h//2, 2, w//2, 2).mean(axis=(1,3))
print('\n2x2 pooled (6x6 -> 3x3):\n', pooled.astype(int))

In [ ]:
import numpy as np
np.random.seed(42)

# FINANCIAL TIME SERIES: Safaricom Stock
days = 252
prices = np.zeros(days)
prices[0] = 22.0
for i in range(1, days):
    prices[i] = prices[i-1] * (1 + np.random.normal(0.0005, 0.02))
prices = np.round(prices, 2)

# Moving Averages
def sma(arr, w):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

sma20 = sma(prices, 20)
sma50 = sma(prices, 50)

print(f'Final price (day 252): KES {prices[-1]:.2f}')
print(f'SMA-20 on day 252  : KES {sma20[-1]:.2f}')
print(f'SMA-50 on day 252  : KES {sma50[-1]:.2f}')

# Daily returns
returns = np.diff(prices) / prices[:-1]
print(f'\nMean daily return: {returns.mean()*100:.3f}%')
print(f'Daily volatility : {returns.std()*100:.3f}%')
print(f'Annualised vol   : {returns.std()*np.sqrt(252)*100:.2f}%')

# Signal: buy when price > SMA20, sell when below
aligned_sma = sma20[-(len(prices)-20):]  # align lengths
recent_prices = prices[20:]
signal = np.where(recent_prices > aligned_sma, 'BUY', 'SELL')
buy_days = (signal == 'BUY').sum()
print(f'\nBuy signal days: {buy_days}, Sell signal days: {len(signal)-buy_days}')

In [ ]:
import numpy as np
np.random.seed(99)

# FULL ML PREPROCESSING PIPELINE
n, p = 300, 5

# Raw data with issues
X_raw = np.random.randn(n, p) * np.array([10, 100, 0.5, 1000, 0.1])
# Inject 10% missing values in column 0
missing_idx = np.random.choice(n, 30, replace=False)
X_raw[missing_idx, 0] = np.nan

print(f'Input shape: {X_raw.shape}')
print(f'NaN count  : {np.isnan(X_raw).sum()}')

# 1. Median imputation
medians = np.nanmedian(X_raw, axis=0)
for j in range(p):
    mask = np.isnan(X_raw[:, j])
    X_raw[mask, j] = medians[j]

# 2. Z-score normalisation
mean = X_raw.mean(axis=0)
std  = X_raw.std(axis=0)
X_scaled = (X_raw - mean) / std

# 3. Clip outliers
X_clipped = np.clip(X_scaled, -3, 3)

print(f'\nAfter pipeline:')
print(f'NaN remaining  : {np.isnan(X_clipped).sum()}')
print(f'Column means   : {X_clipped.mean(axis=0).round(3)}')
print(f'Column stds    : {X_clipped.std(axis=0).round(3)}')

## Exercise: Logistic Regression from Scratch

Using NumPy only (no sklearn), implement logistic regression with gradient descent.

In [ ]:
import numpy as np
np.random.seed(42)

# Binary classification dataset
X = np.random.randn(200, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(float)

# Normalise
X = (X - X.mean(0)) / X.std(0)

# Implement:
# sigmoid(z) = 1 / (1 + exp(-z))
# forward: p = sigmoid(X @ w + b)
# loss: BCE = -mean(y*log(p) + (1-y)*log(1-p))
# grad: dw = X.T @ (p - y) / n
#            db = mean(p - y)
# Update: w -= lr * dw;  b -= lr * db

# YOUR CODE HERE

## Summary

- NumPy enables image processing without any library
- Rolling windows are implemented with `np.convolve`
- A complete preprocessing pipeline fits in ~10 lines of NumPy
- Logistic regression is implementable with just matrix ops

**Next: Day 12 — NumPy File I/O**